# Shor's Factoring Algorithm

Shor's algorithm factors integers efficiently using quantum phase estimation to find the period of modular exponentiation. The quantum part estimates the phase, while classical post-processing extracts factors using continued fractions and GCD.

We factor $N = 15$ using coprime $a = 13$.

In [ ]:
import qsharp
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction

In [ ]:
qsharp.init()
with open("Program.qs") as f:
    qsharp.eval(f.read())

from qsharp.code import EstimatePhase

In [ ]:
def period_from_phase(phase_result, N):
    """Use continued fractions to extract the period from a phase measurement."""
    precision_bits = 2 * N.bit_length()
    if phase_result == 0:
        return 0
    frac = Fraction(phase_result, 2**precision_bits).limit_denominator(N)
    return abs(frac.denominator)

def find_factors(a, N, r):
    """Given period r, attempt to find non-trivial factors of N."""
    if r == 0 or r % 2 != 0:
        return []
    half_r = pow(a, r // 2, N)
    if half_r == N - 1:
        return []
    candidates = [gcd(half_r + 1, N), gcd(half_r - 1, N)]
    return [f for f in candidates if 1 < f < N and N % f == 0]

## Run factoring

Each attempt runs QPE once (quantum), then classically extracts the period and checks for factors.

In [ ]:
N = 15
a = 13
max_attempts = 10
precision_bits = 2 * N.bit_length()

phases = []
factors_found = None

for attempt in range(1, max_attempts + 1):
    phase_result = EstimatePhase(a, N)
    phase_value = phase_result / 2**precision_bits
    period = period_from_phase(phase_result, N)
    factors = find_factors(a, N, period)
    phases.append(phase_value)

    status = f"factors: {factors}" if factors else "no factors"
    print(f"Attempt {attempt}: phase = {phase_value:.4f}, period = {period}, {status}")

    if factors:
        factors_found = factors
        break

if factors_found:
    print(f"\nNon-trivial factors of {N}: {factors_found}")
else:
    print(f"\nFailed to find factors after {max_attempts} attempts.")

## Phase distribution over many runs

Running QPE many times shows the distribution of measured phases. Peaks correspond to multiples of $1/r$ where $r$ is the period.

In [ ]:
shots = 100
phase_results = [EstimatePhase(a, N) for _ in range(shots)]
phase_values = [r / 2**precision_bits for r in phase_results]

fig, ax = plt.subplots(figsize=(10, 4))
counts = {}
for v in phase_values:
    v_rounded = round(v, 4)
    counts[v_rounded] = counts.get(v_rounded, 0) + 1

sorted_items = sorted(counts.items())
labels = [f"{k:.4f}" for k, _ in sorted_items]
values = [v for _, v in sorted_items]
ax.bar(labels, values)
ax.set_xlabel("Measured phase")
ax.set_ylabel("Counts")
ax.set_title(f"Phase distribution for factoring {N} (a={a}, {shots} shots)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()